# Phase 2: Multi-Model Machine Learning Engine & PD Benchmarking
### Point-in-Time (PiT) Credit Risk Modeling & Macro Transmission

**Objective:**  
Train and evaluate a **4-Model Champion-Challenger Credit Risk Benchmark** to predict borrower Probability of Default (PD):
1. **Logistic Regression (Scorecard Baseline):** Regulatory baseline under Basel/IFRS 9 guidelines.
2. **Random Forest Classifier:** Bagging ensemble capturing non-linear interactions via bootstrap aggregation.
3. **LightGBM Classifier:** Fast leaf-wise histogram gradient boosting.
4. **XGBoost Classifier:** Exact depth-wise histogram gradient boosting.

---

### Mathematical Definition
$$\text{PD}_i(t) = P(Y_i = 1 \mid \mathbf{X}_i, \mathbf{M}_t) = \sigma\left(f(\text{FICO}_i, \text{DTI}_i, \text{Purpose}_i, \text{UNRATE}_t, \text{FEDFUNDS}_t)\right)$$

---

### Hardware Optimization (Intel i3 / 8GB RAM)
- **Downcasting:** Features cast to `float32` (< 250 MB RAM footprint).
- **Parallel Execution:** Multi-threading across available cores (`n_jobs=-1`).
- **Out-of-Time (OOT) Split:** Chronological split (Train: 2007–2015, Test: 2016–2018) ensuring zero lookahead leakage.

In [ ]:
# Cell 1: Imports & Setup
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, 
    log_loss, 
    brier_score_loss, 
    roc_curve, 
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

print("All 4 ML libraries (Sklearn, RandomForest, LightGBM, XGBoost) loaded successfully.")

In [ ]:
# Cell 2: Data Loading & Memory Optimization
data_path = "data/cleaned_loans_phase1.parquet"
df = pd.read_parquet(data_path)

# Downcast datatypes to float32 / int8 for fast computation and low memory
df['loan_amnt'] = df['loan_amnt'].astype(np.float32)
df['fico_range_low'] = df['fico_range_low'].astype(np.float32)
df['dti'] = df['dti'].astype(np.float32)
df['UNRATE'] = df['UNRATE'].astype(np.float32)
df['FEDFUNDS'] = df['FEDFUNDS'].astype(np.float32)
df['target'] = df['target'].astype(np.int8)
df['purpose'] = df['purpose'].astype('category')

mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Loaded {len(df):,} loans. In-Memory Footprint: {mem_mb:.2f} MB (Extremely safe for 8GB RAM)")

In [ ]:
# Cell 3: Out-of-Time (OOT) Chronological Split
# Train: 2007-2015 vintages | Test (OOT): 2016-2018 vintages
train_mask = df['Year_Month'] < '2016-01'
test_mask = df['Year_Month'] >= '2016-01'

features = ['fico_range_low', 'dti', 'purpose', 'UNRATE', 'FEDFUNDS']
target_col = 'target'

X_train = df.loc[train_mask, features].copy()
y_train = df.loc[train_mask, target_col].values

X_test = df.loc[test_mask, features].copy()
y_test = df.loc[test_mask, target_col].values

print(f"Training Cohort (2007-2015): {len(X_train):,} loans (Default Rate: {y_train.mean()*100:.2f}%)")
print(f"OOT Test Cohort (2016-2018): {len(X_test):,} loans (Default Rate: {y_test.mean()*100:.2f}%)")

In [ ]:
# Cell 4: Feature Preprocessor Pipeline
num_cols = ['fico_range_low', 'dti', 'UNRATE', 'FEDFUNDS']
cat_cols = ['purpose']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

X_train_proc = preprocessor.fit_transform(X_train).astype(np.float32)
X_test_proc = preprocessor.transform(X_test).astype(np.float32)
print(f"Preprocessed feature shape: {X_train_proc.shape}")

In [ ]:
# Cell 5: Credit Risk Metric Evaluation Helper
def evaluate_credit_model(name, y_true, y_prob, fit_time_sec):
    auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    gini = 2 * auc - 1
    loss = log_loss(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ks_stat = np.max(tpr - fpr)
    
    return {
        'Model': name,
        'ROC-AUC': round(auc, 4),
        'Gini Coefficient': round(gini, 4),
        'KS Statistic (%)': round(ks_stat * 100, 2),
        'PR-AUC': round(pr_auc, 4),
        'Log-Loss': round(loss, 4),
        'Brier Score': round(brier, 4),
        'Training Time (s)': round(fit_time_sec, 2)
    }

results = []
model_probs = {}

In [ ]:
# Cell 6: Model 1 - Logistic Regression (Scorecard Baseline)
print("Training Model 1: Logistic Regression...")
t0 = time.time()
lr = LogisticRegression(max_iter=500, solver='lbfgs', random_state=42)
lr.fit(X_train_proc, y_train)
t_lr = time.time() - t0

y_prob_lr = lr.predict_proba(X_test_proc)[:, 1]
model_probs['Logistic Regression'] = y_prob_lr
res_lr = evaluate_credit_model("Logistic Regression (Scorecard Baseline)", y_test, y_prob_lr, t_lr)
results.append(res_lr)
print(f"LR Done in {t_lr:.2f}s | AUC: {res_lr['ROC-AUC']} | KS: {res_lr['KS Statistic (%)']}%")

In [ ]:
# Cell 7: Model 2 - Random Forest Classifier (Optimized Bagging Ensemble)
print("Training Model 2: Random Forest Classifier...")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    max_samples=0.3,       # Subsample per tree for rapid execution & memory protection
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_proc, y_train)
t_rf = time.time() - t0

y_prob_rf = rf.predict_proba(X_test_proc)[:, 1]
model_probs['Random Forest'] = y_prob_rf
res_rf = evaluate_credit_model("Random Forest Classifier", y_test, y_prob_rf, t_rf)
results.append(res_rf)
print(f"Random Forest Done in {t_rf:.2f}s | AUC: {res_rf['ROC-AUC']} | KS: {res_rf['KS Statistic (%)']}%")

In [ ]:
# Cell 8: Model 3 - LightGBM Classifier (Fast Histogram Gradient Boosting)
print("Training Model 3: LightGBM Classifier...")
t0 = time.time()
lgbm = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.08,
    num_leaves=31,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)
lgbm.fit(X_train_proc, y_train)
t_lgbm = time.time() - t0

y_prob_lgbm = lgbm.predict_proba(X_test_proc)[:, 1]
model_probs['LightGBM'] = y_prob_lgbm
res_lgbm = evaluate_credit_model("LightGBM Classifier", y_test, y_prob_lgbm, t_lgbm)
results.append(res_lgbm)
print(f"LightGBM Done in {t_lgbm:.2f}s | AUC: {res_lgbm['ROC-AUC']} | KS: {res_lgbm['KS Statistic (%)']}%")

In [ ]:
# Cell 9: Model 4 - XGBoost Classifier (Histogram Tree Method)
print("Training Model 4: XGBoost Classifier...")
t0 = time.time()
xgb_clf = xgb.XGBClassifier(
    tree_method='hist',
    n_estimators=150,
    learning_rate=0.08,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_clf.fit(X_train_proc, y_train)
t_xgb = time.time() - t0

y_prob_xgb = xgb_clf.predict_proba(X_test_proc)[:, 1]
model_probs['XGBoost'] = y_prob_xgb
res_xgb = evaluate_credit_model("XGBoost (Hist Tree Ensemble)", y_test, y_prob_xgb, t_xgb)
results.append(res_xgb)
print(f"XGBoost Done in {t_xgb:.2f}s | AUC: {res_xgb['ROC-AUC']} | KS: {res_xgb['KS Statistic (%)']}%")

In [ ]:
# Cell 10: 4-Model Leaderboard & ROC Curve Comparison
df_leaderboard = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True)
print("=" * 75)
print("4-MODEL CREDIT RISK BENCHMARK LEADERBOARD (OOT Test Set):")
print("=" * 75)
display(df_leaderboard)

# Visualizing Comparison Curves
plt.figure(figsize=(9, 6))
for name, probs in model_probs.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Out-of-Time (OOT) ROC Curves Comparison (4 Models)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 11: Export Champion Model Pipeline & PD Baseline for Phase 3
os.makedirs("models", exist_ok=True)

# Save XGBoost Pipeline
champion_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_clf)
])
joblib.dump(champion_pipeline, "models/champion_pd_model.joblib")
print("Saved Champion Model Pipeline to models/champion_pd_model.joblib")

# Save Test Portfolio with Baseline PD
df_test_portfolio = df.loc[test_mask].copy()
df_test_portfolio['PD_base'] = y_prob_xgb.astype(np.float32)
df_test_portfolio.to_parquet("data/test_portfolio_with_pd.parquet", index=False)
print(f"Saved Test Portfolio ({len(df_test_portfolio):,} loans) to data/test_portfolio_with_pd.parquet")